# 04 — Cuarto estudio: versión paralela multi-b

Este notebook usa `joblib` para paralelizar réplicas Monte Carlo por escenario y tamaño de bloque. La configuración inicial es pequeña.

In [ ]:
# Ejecutar una vez si aparece ModuleNotFoundError
from pathlib import Path
import sys, subprocess
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
ROOT = None
for path in candidates:
    if (path / "pyproject.toml").exists() and (path / "src" / "codispersion_bootstrap").exists():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError(f"No encontré la carpeta raíz del proyecto. Estoy parado en: {cwd}")
print("Carpeta raíz encontrada:", ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(ROOT)])
print("Paquete instalado correctamente. Reinicia el kernel después de esta celda si es primera vez.")

In [1]:
from codispersion_bootstrap import build_H, run_study_multi_b_parallel_to_csv

## Test

In [6]:
out_test = run_study_multi_b_parallel_to_csv(
    n1=64, n2=64,
    H=build_H(hmax=2),
    rhos=(0.2, ),
    ranges=(3.0, ),
    anis_modes=("iso", "aniso"),
    Bsizes=(57,),   # ojo: tupla con coma
    B=120,
    R=10,
    n_jobs=-1,
    path_detailed_csv="../results/cuarto_estudio_detailed_quick.csv",
    path_summary_csv="../results/cuarto_estudio_summary_quick.csv",
)
out_test["summary"].head()

[1/2] rho0=0.2, range=3.0, anis=iso, b=57, R=10, B=120
[2/2] rho0=0.2, range=3.0, anis=aniso, b=57, R=10, B=120


,scenario_id,rho0,range_pix,anis_mode,b,h,h1,h2,rho_hat_mean,rho_hat_var_mc,var_boot_mean,coverage,width_mean,R
0,0,0.2,3.0,iso,57,"(1, 0)",1,0,0.204301,0.000765,0.000327,0.7,0.067460,10
1,0,0.2,3.0,iso,57,"(0, 1)",0,1,0.184283,0.000369,0.000316,0.7,0.066574,10
2,0,0.2,3.0,iso,57,"(1, 1)",1,1,0.197698,0.000557,0.000301,0.9,0.066146,10
3,0,0.2,3.0,iso,57,"(-1, 1)",-1,1,0.191635,0.000918,0.000357,0.8,0.071439,10
4,0,0.2,3.0,iso,57,"(2, 0)",2,0,0.203393,0.001205,0.000383,0.6,0.074794,10


In [8]:
out_test

{'detailed':      scenario_id  rep  rho0  range_pix anis_mode   b        h  h1  h2  \
 0              0    0   0.2        3.0       iso  57   (1, 0)   1   0   
 1              0    0   0.2        3.0       iso  57   (0, 1)   0   1   
 2              0    0   0.2        3.0       iso  57   (1, 1)   1   1   
 3              0    0   0.2        3.0       iso  57  (-1, 1)  -1   1   
 4              0    0   0.2        3.0       iso  57   (2, 0)   2   0   
 ..           ...  ...   ...        ...       ...  ..      ...  ..  ..   
 155            1    9   0.2        3.0     aniso  57  (-1, 1)  -1   1   
 156            1    9   0.2        3.0     aniso  57   (2, 0)   2   0   
 157            1    9   0.2        3.0     aniso  57   (0, 2)   0   2   
 158            1    9   0.2        3.0     aniso  57   (2, 2)   2   2   
 159            1    9   0.2        3.0     aniso  57  (-2, 2)  -2   2   
 
       rho_hat  var_boot     ci_lo     ci_hi  
 0    0.194362  0.000318  0.158815  0.222372  
 1  

## Simulación

In [ ]:
out = run_study_multi_b_parallel_to_csv(
    n1=256, n2=256, H=build_H(hmax=2),
    rhos=[0.2, 0.5, 0.8],
    ranges=[3.0, 7.0, 12.0],
    anis_modes=["iso","aniso"],
    Bsizes=[57],   # menos b para reducir carga
    B=400,               # bootstrap moderado
    R=100,               # MC moderado
    nu=1.0, angle_deg=45.0, anis_ratio=2.0,
    base_seed=2025,
    n_jobs=-1,      
    path_detailed_csv="../results/cuarto_estudio_detailed_quick.csv",
    path_summary_csv="../results/cuarto_estudio_summary_quick.csv",
)
out["summary"].head()

In [ ]:
out["summary"]